In [ ]:
import plotly.express as px
import pandas as pd
import struct
from glob import glob
from os.path import basename

#plot_order = ["blis", "strassen", "ozimmu_slice_max18", "openblas", "ozimmu_slice_min3", "ozimmu_auto_thresh0", "netlibblas", "ozimmu_auto_thresh8", "ozimmu_auto_thresh1", "cublas", "ozimmu_auto_thresh4", "ozimmu_auto_thresh2"]
plot_order = ["blis", "strassen", "openblas", "ozimmu_auto_thresh0", "netlibblas", "cublas"]
chars = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L"]
cached_dfs = {}

def summarize_all_componentwise_signed_relative_errors(matrix_type):

    # get paths to all error files
    filepaths = glob(f"results/{matrix_type}/all_componentwise_signed_absolute_errors.*.bin")

    # create a single dataframe for all data
    dfs = []
    reference_values = []
    for filepath in filepaths:
        name = filepath.split(".")[-2]
        if name not in plot_order:
            continue

        if filepath in cached_dfs:
            dfs.append(cached_dfs[filepath])
        else:
            # read in reference values
            if len(reference_values) == 0:
                with open(f"results/{matrix_type}/reference_values.bin", "rb") as f:
                    byte_buffer = f.read()
                    for i in range(0, len(byte_buffer), 8):
                        reference_values.append(struct.unpack('d', byte_buffer[i:i+8]))
                reference_values = pd.DataFrame(reference_values, columns=['reference_values'])    
    
            with open(filepath, "rb") as f:
                byte_buffer = f.read()
    
            signed_absolute_errors = []
            start_byte = 0#8*(32*32)
            end_byte = len(byte_buffer)#8*(32*32 + 64*64)
            for i in range(start_byte,end_byte,8):
                signed_absolute_errors.append(struct.unpack('d', byte_buffer[i:i+8]))
    
            df = pd.DataFrame(signed_absolute_errors, columns=['signed_absolute_errors'])
            df["name"] = name
            df["plot_order"] = f"Figure {chars[plot_order.index(name)]} - {name}"
            df["signed_relative_errors"] = df["signed_absolute_errors"] / reference_values["reference_values"]
            dfs.append(df)

            # save the data
            cached_dfs[filepath] = df

    df = pd.concat(dfs)

    print(f"\n\n============= summary statistics of all signed RELATIVE errors for {matrix_type} matrices =============\n\n")
    for filepath in sorted(filepaths):
        name = filepath.split(".")[-2]
        if name not in plot_order:
            continue
        df_slice = df[df['name'] == name]
        print(f"{name} stats:")
        print(f"\t   mean of signed relative errors: {df_slice['signed_relative_errors'].mean()}")
        print(f"\t    std of signed relative errors: {df_slice['signed_relative_errors'].std()}")

    print(f"\n\n============= histograms of signed RELATIVE errors for {matrix_type} matrices =============\n\n")
    hist = df.hist(column="signed_relative_errors", by='plot_order', figsize=(20,10), bins=120, xrot=0)


def summarize_all_componentwise_signed_absolute_errors(matrix_type):

    # get paths to all error files
    filepaths = glob(f"results/{matrix_type}/all_componentwise_signed_absolute_errors.*.bin")

    # create a single dataframe for all data
    dfs = []
    reference_values = []
    for filepath in filepaths:
        name = filepath.split(".")[-2]
        if name not in plot_order:
            continue

        if filepath in cached_dfs:
            dfs.append(cached_dfs[filepath])
        else:
            if len(reference_values) == 0:
                # read in reference values
                with open(f"results/{matrix_type}/reference_values.bin", "rb") as f:
                    byte_buffer = f.read()
                    for i in range(0, len(byte_buffer), 8):
                        reference_values.append(struct.unpack('d', byte_buffer[i:i+8]))
                reference_values = pd.DataFrame(reference_values, columns=['reference_values'])    
    
            with open(filepath, "rb") as f:
                byte_buffer = f.read()
    
            signed_absolute_errors = []
            start_byte = 0#8*(32*32)
            end_byte = len(byte_buffer)#8*(32*32 + 64*64)
            for i in range(start_byte,end_byte,8):
                signed_absolute_errors.append(struct.unpack('d', byte_buffer[i:i+8]))
    
            df = pd.DataFrame(signed_absolute_errors, columns=['signed_absolute_errors'])
            df["name"] = name
            df["plot_order"] = f"Figure {chars[plot_order.index(name)]} - {name}"
            df["signed_relative_errors"] = df["signed_absolute_errors"] / reference_values["reference_values"]
            dfs.append(df)

            # save the data
            cached_dfs[filepath] = df

    df = pd.concat(dfs)
    print(f"\n\n============= summary statistics of all signed ABSOLUTE errors for {matrix_type} matrices =============\n\n")
    for filepath in sorted(filepaths):
        name = filepath.split(".")[-2]
        if name not in plot_order:
            continue
        df_slice = df[df['name'] == name]
        print(f"{name} stats:")
        print(f"\t   mean of signed absolute errors: {df_slice['signed_absolute_errors'].mean()}")
        print(f"\t    std of signed absolute errors: {df_slice['signed_absolute_errors'].std()}")
        
    print(f"\n\n============= histograms of all signed ABSOLUTE errors for {matrix_type} matrices =============\n\n")
    hist = df.hist(column="signed_absolute_errors", by='plot_order', figsize=(20,10), bins=120, xrot=0)


def plot_max_componentwise_unsigned_relative_errors_vs_dim(matrix_type, log_x=False, log_y=False):

    filepaths = glob(f"results/{matrix_type}/max_componentwise_unsigned_errors.*.txt")
    
    dfs = []
    for filepath in filepaths:
        df = pd.read_csv(filepath)
        name = filepath.split(".")[-2]
        if name not in plot_order:
            continue
        df["name"] = name
        dfs.append(df)
    
    df = pd.concat(dfs, ignore_index=True)

    fig = px.line(
        df,
        x="n",
        y="max_componentwise_unsigned_relative_error",
        markers=True,
        log_x=log_x,
        log_y=log_y,
        title=f"Max Componentwise Unsigned Relative Error vs dim for {matrix_type} matrices",
        hover_data=["m", "n", "k"],
        symbol = "name",
        color="name",
        color_discrete_sequence=px.colors.qualitative.Safe,
    )
    fig.update_traces(
        marker=dict(
            size=20,
            opacity=0.7,
            line_width=2.0,
        )
    )
    fig.update_layout(
        yaxis_tickformat='.2e',
        #xaxis_tickformat='.2e'
        width=1000,
        height=500
    )

    fig.show()

def plot_max_componentwise_unsigned_absolute_errors_vs_dim(matrix_type, log_x=False, log_y=False):

    filepaths = glob(f"results/{matrix_type}/max_componentwise_unsigned_errors.*.txt")
    
    dfs = []
    for filepath in filepaths:
        df = pd.read_csv(filepath)
        name = filepath.split(".")[-2]
        if name not in plot_order:
            continue
        df["name"] = name
        dfs.append(df)
    
    df = pd.concat(dfs, ignore_index=True)

    fig = px.line(
        df,
        x="n",
        y="max_componentwise_unsigned_absolute_error",
        markers=True,
        log_x=log_x,
        log_y=log_y,
        title=f"Max Componentwise Unsigned Absolute Error vs dim for {matrix_type} matrices",
        hover_data=["m", "n", "k"],
        symbol = "name",
        color="name",
        color_discrete_sequence=px.colors.qualitative.Safe,
    )
    fig.update_traces(
        marker=dict(
            size=20,
            opacity=0.7,
            line_width=2.0,
        )
    )
    fig.update_layout(
        yaxis_tickformat='.2e',
        #xaxis_tickformat='.2e'
        width=1000,
        height=500
    )

    fig.show()

def analyze_n_zeros(ratio):
    filepaths = glob(f"results/test1/max_componentwise_abs_errors.*_{ratio}.txt")

    ratio_string = ratio[0] + "." + ratio[1:]

    ratio_string = ratio_string.split("-")[0] + ", worstcase selection of zero rows/cols"
    
    dfs = []
    for filepath in filepaths:
        df = pd.read_csv(filepath)
        df["filename"] = basename(filepath)
        name = filepath.split(".")[-2].split("_")[0]
        df["name"] = name
        dfs.append(df)
    
    df = pd.concat(dfs, ignore_index=True)

    for filepath in filepaths:
        df_slice = df[df['filename'] == basename(filepath)]
        print(f"{filepath}:")
        print(f"\tn_zeros: {df_slice['n_zeros'].sum()}\n")

    df2 = df.copy()

    ref = df[df["filename"] == f"max_componentwise_abs_errors.netlibblas_{ratio}.txt"][["n", "n_zeros"]].rename(columns={"n_zeros": "ref_n_zeros"})

    df2 = df2.merge(ref, on="n", how="left")

    df2["% of zero entries lost"] = (1 - df2["n_zeros"] / df2["ref_n_zeros"]) * 100

    fig = px.box(
        df2[df2["filename"] == f"max_componentwise_abs_errors.strassen_{ratio}.txt"],
        x = "n",
        y = "% of zero entries lost",
        log_x = True,
        title=f"Percentage of zeros lost vs problem dimension<br>(z_ratio = {ratio_string})<br>",
        points = "all",
    )
    
    #fig = px.line(
    #    df2[df2["name"] == "strassen"],
    #    x="n",
    #    y="% of zero entries retained",
    #    markers=True,
    #    title=f"Percentage of zeros retained vs problem dimension",
    #    hover_data=["m", "n", "k"],
        #symbol = "name",
        #symbol="Square?",
        #symbol_sequence=["square", "arrow-wide"],
        #color="name",
        #color_discrete_sequence=px.colors.qualitative.Safe,
    #)
    #fig.update_traces(
    #    marker=dict(
    #        size=20,
    #        opacity=0.7,
    #        line_width=2.0,
    #    )
    #)
    fig.update_layout(
        xaxis_title = "Dimension",
        yaxis_title = "Zeros Lost",
        yaxis_ticksuffix = "%",
        #yaxis_tickformat='.2e',
        #xaxis_tickformat='.2e'
        yaxis_range=[0,100],
        width=1000,
        height=500
    )

    fig = postprocess_figure_for_blog(fig)
    fig.show()

def postprocess_figure_for_blog(fig):

    fig.update_layout(paper_bgcolor='#f8f8f8',
                        font_size=18,
                        font_family="Serif",
                        width=1000,
                        height=500)
    return fig


# Test 1 Matrices: Discriminating Between $O(n^3)$ and Strassen
## Setup
- Problem Sizes: 32, 64, 128, 256, 512, 1024, 2048
- Error Metric: counting total number of zeros in calculated C matrices across all problem sizes
- Tested Implementations:
  1. netlibBLAS
  2. BLIS
  3. OpenBLAS
  4. cuBLAS
  5. OZIMMU with auto slices, mantissa loss threshold of 0 bits
  6. naive 1-level Strassen
## Results, Summarized
Works as expected; see counts below.

In [ ]:
#analyze_n_zeros("025")
analyze_n_zeros("025-worstcase")

# Test 3 Matrices: Discriminating Between $O(n^3)$+FP and $O(n^3)$+emulation
## Setup
- Problem Sizes: 32, 64, 128, 256, 512, 1024, 2048
- Error Metrics:
  1. All componentwise errors across all problem sizes
  2. Max componentwise error for each problem size
- Tested Implementations:
    1. netlibBLAS
    2. cuBLAS
    3. BLIS
    4. OpenBLAS
    5. Naive 1-level Strassen
    6. OZIMMU with auto slices, mantissa loss threshold of 0 bits
- Input Matrices:
  1. Baseline 1: A,B ~ U(0,1)
  2. Baseline 2: A,B ~ U(-1,1)
  3. Treatment: test3 matrices

## Baseline 1: A,B ~ U(0,1)

In [ ]:
plot_max_componentwise_unsigned_relative_errors_vs_dim("U_0_1")
plot_max_componentwise_unsigned_absolute_errors_vs_dim("U_0_1")

In [ ]:
summarize_all_componentwise_signed_relative_errors("U_0_1")

In [ ]:
summarize_all_componentwise_signed_absolute_errors("U_0_1")

## Baseline 2: A,B ~ U(-1,1)

In [ ]:
plot_max_componentwise_unsigned_relative_errors_vs_dim("U_-1_1")
plot_max_componentwise_unsigned_absolute_errors_vs_dim("U_-1_1")

In [ ]:
summarize_all_componentwise_signed_relative_errors("U_-1_1")

In [ ]:
summarize_all_componentwise_signed_absolute_errors("U_-1_1")

## Treatment: Test3 Matrices

In [ ]:
plot_max_componentwise_unsigned_relative_errors_vs_dim("test3")
plot_max_componentwise_unsigned_absolute_errors_vs_dim("test3")

In [ ]:
summarize_all_componentwise_signed_relative_errors("test3")

In [ ]:
summarize_all_componentwise_signed_absolute_errors("test3")